## Path B - regenerate the result files from scratch (optional, GPU)

Not needed to verify the paper. The cell below re-runs the six experiment notebooks that
live inside the bundle (in `notebooks/`), one at a time, writing fresh outputs into
`results/`. Then re-run the Path A cell above to re-check the regenerated files.

**Requirements:** set a GPU runtime first (**Runtime > Change runtime type > GPU**). The
three synthetic experiments need no data; the three real-data experiments
(`experiment_beijing`, `experiment_rv`, `experiment_wdi_resource_curse`) need their
dataset placed in the unpacked bundle's `data/` folder (public sources are listed in
`data/README.md`); those will error cleanly if the data file is absent, which is expected.

Re-runs reproduce the exact-checked quantities (recovery counts, parameter counts, rank
orderings). Seed- and hardware-sensitive quantities (held-out MSEs, cross-fit margins,
seed-agreement fractions) are verified as ranges and may differ slightly.

In [ ]:
# Re-run every experiment notebook from the unpacked bundle, one at a time.
# Each notebook regenerates its own results/<experiment>/ folder.
import os, sys, subprocess

NOTEBOOK_ORDER = [
    'experiment1.ipynb',                    # synthetic recovery (no data, no GPU strictly needed)
    'experiment2.ipynb',                    # support-collapse rho-sweep (no data)
    'experiment_gating_value.ipynb',        # capacity-matched baselines (no data)
    'experiment_beijing.ipynb',             # needs data/beijing_multisite.csv
    'experiment_rv.ipynb',                  # needs data/rv_dataset.csv
    'experiment_wdi_resource_curse.ipynb',  # needs data/wdi_reversal_panel.csv
]
NB_DIR = os.path.join(ROOT, 'notebooks')
assert os.path.isdir(NB_DIR), f'notebooks/ not found under {ROOT}'

# nbconvert ships with Colab; execute each notebook in place (run from ROOT so paths resolve).
for name in NOTEBOOK_ORDER:
    path = os.path.join(NB_DIR, name)
    if not os.path.exists(path):
        print(f'SKIP (missing): {name}'); continue
    print(f'\n{"="*70}\nRunning {name}\n{"="*70}', flush=True)
    cmd = [sys.executable, '-m', 'nbconvert', '--to', 'notebook', '--execute',
           '--inplace', '--ExecutePreprocessor.timeout=-1', path]
    proc = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if proc.returncode == 0:
        print(f'  OK: {name} completed; results written to results/')
    else:
        # Most likely cause: a real-data notebook whose dataset is not present.
        tail = (proc.stderr or proc.stdout).strip().splitlines()[-6:]
        print(f'  {name} did not finish. Last lines:')
        for ln in tail: print('   ', ln)
        print('  (For real-data notebooks this usually means the dataset is not in data/.)')

print('\nDone. Re-run the Path A audit cell above to verify the regenerated results.')

## Path A — audit every paper number

In [ ]:
# Upload the bundle, then it unpacks automatically.
# Running this cell shows a 'Choose Files' button -- select gnavar-icdm-reproducibility.tar.gz.
import os, glob, tarfile, sys

def _find_tarball():
    for pat in ['/content/*reproducibility*.tar.gz', '/content/*.tar.gz', '*.tar.gz']:
        hits = glob.glob(pat)
        if hits: return hits[0]
    return None

TARBALL = _find_tarball()
if TARBALL is None:
    try:
        from google.colab import files
        print('Click "Choose Files" and select gnavar-icdm-reproducibility.tar.gz')
        uploaded = files.upload()          # renders the upload button
        names = [n for n in uploaded if n.endswith('.tar.gz')]
        assert names, 'Please upload the .tar.gz bundle.'
        TARBALL = os.path.abspath(names[0])  # files.upload() writes to cwd; make path absolute
    except ImportError:
        raise FileNotFoundError(
            'Not running in Colab and no .tar.gz found in the working directory. '
            'Place gnavar-icdm-reproducibility.tar.gz here and re-run.')
print('Using bundle:', TARBALL)

EXTRACT_TO = '/content/gnavar_repro' if os.path.isdir('/content') else os.path.expanduser('~/gnavar_repro')
os.makedirs(EXTRACT_TO, exist_ok=True)
with tarfile.open(TARBALL) as t:
    t.extractall(EXTRACT_TO)

ROOT = None
for dp, dn, fn in os.walk(EXTRACT_TO):
    if 'results' in dn and 'src' in dn:
        ROOT = dp; break
assert ROOT, 'Could not find the bundle root (a folder with results/ and src/) in the tarball.'
print('Unpacked bundle root:', ROOT)
!pip install -q numpy pandas >/dev/null 2>&1
print('Dependencies ready.')

In [ ]:
# === Cell 2: recompute and check every paper number ===
import sys
sys.path.insert(0, ROOT + '/src')
from verifier_core import CHECKS

def _cmp(kind, expected, got, tol):
    if kind == 'exact':  return got == expected
    if kind == 'tol':    return abs(float(got) - float(expected)) <= tol
    if kind == 'range':
        lo, hi = expected
        return (lo <= got[0] and got[1] <= hi) if isinstance(got,(tuple,list)) else (lo <= got <= hi)
    raise ValueError(kind)

def _fmt(v):
    if isinstance(v, float): return f'{v:.4f}'
    if isinstance(v, (tuple, list)): return '(' + ', '.join(_fmt(x) for x in v) + ')'
    return str(v)

n_pass = 0; fails = []; sec = None
for entry in CHECKS:
    section, quantity, kind, expected, fn = entry[:5]
    tol = entry[5] if len(entry) > 5 else 0.0
    if section != sec: print(f'\n[Section {section}]'); sec = section
    try:
        got = fn(ROOT); ok = _cmp(kind, expected, got, tol)
    except Exception as e:
        got = f'ERROR: {e}'; ok = False
    print(f"  [{'PASS' if ok else 'FAIL'}] {quantity:<52} paper={_fmt(expected)}  artifact={_fmt(got)}")
    if ok: n_pass += 1
    else: fails.append((section, quantity, expected, got))

print('\n' + '='*70)
print(f'RESULT: {n_pass}/{len(CHECKS)} checks passed.')
if fails:
    print('FAILED:'); [print(f'  [{s}] {q}: paper={_fmt(e)} artifact={_fmt(g)}') for s,q,e,g in fails]
else:
    print('All paper numbers reproduce from the committed artifacts.')